# Chapter 48. Big Data & Distributed Compute

*The runnable half of this chapter.* Every code cell below is the chapter's own code, and the words around it are the chapter's own explanation. Run the cells in order: each one uses names made by the cells before it.

Riverstone Supplies is the single worked example throughout the book, so the data here is the same data you meet in every other chapter.

Built from `manuscript/ch48-big-data-and-distributed-compute.md`.

# Chapter 48. Big Data & Distributed Compute

*Part 5 — Data Engineering, Integration & Scale*

> **Chapter at a glance**
>
> **You will learn to:** install Spark and Polars and start your first Spark session · judge for yourself whether a dataset needs distributed computing at all · explain partitions, workers, the driver, and why moving data between machines is what costs · write PySpark that reads, filters, groups, joins, and writes Riverstone's sensor data · read a physical query plan with `explain()` and find the shuffle in it · use partition pruning, column pruning, and broadcast joins, and see each one change the plan · recognize skew and know the usual fixes · use Polars for the first time · compare DuckDB, Polars, and Spark on the same question and interpret the timings · avoid the common ways a Spark job surprises you · place Databricks, EMR, Dataproc, Fabric, Trino, and Ray on the map.
>
> **Before you start:** Chapter 17 (the virtual environment and Jupyter). Chapter 18 (pandas, and Parquet). Chapter 26 section 26.0 (the terminal). Chapter 29 (Python as software). Chapter 45 (DuckDB from Python, and Parquet). Chapter 46 (pipelines and idempotent re-runs). Chapter 14's regular expressions help with one optional helper in section 48.4.
>
> **Time needed:** 14–18 hours of reading and practice, spread over two to three weeks. Plan four sittings: section 48.0, installing Java, Spark, and Polars and starting your first Spark session (1–2 hours, more if Java fights you); sections 48.1–48.4; sections 48.5–48.6 (making jobs faster, Polars, three engines); section 48.7 onwards and the project.
>
> **Tools:** Python in the book's virtual environment with `pyspark` and `polars` (installed in section 48.0), `duckdb` (Chapter 45), and `pandas` and `pyarrow` (Chapter 18); a Java runtime (Java 17, 21, or 25) for Spark. Everything runs on one machine.
>
> **Practice data:** Riverstone's plant sensor readings, generated by the Chapter 48 companion script: 25 machines across two plants, one reading every 10 seconds from 1 October to 31 December 2025. That's **19,872,000 readings**, about 114 MB as Parquet. It's a larger, separate dataset from Chapter 40's sensor practice data.

---

## Why this matters

Somewhere in every data engineer's first year, a dataset stops fitting comfortably in memory, a job that took two minutes starts taking two hours, and somebody says the word "Spark".

Two mistakes follow. The first is reaching for a cluster far too early, which is expensive, slow to develop on, and harder to debug than the problem it replaced. The second is refusing to learn distributed computing at all, and then writing single-machine code that quietly can't grow, or Spark code that works but costs ten times what it should because of one line.

This chapter gives you the judgment and the mechanics. You'll write real PySpark on 20 million sensor readings, read the plans Spark makes, watch a shuffle appear and then make it disappear, and time the same question in three tools. By the end you should be able to answer, with reasons and numbers, the question every interviewer eventually asks: *"How would you process this, and why not something simpler?"*

---

## In plain English

Think about counting the votes in an election.

- One counting officer can handle a small booth. That's a **single machine**, and it's usually enough.
- For a whole state, you divide ballots into boxes and hand them to many officers. Each box is a **partition**; each officer is a **worker**.
- A supervisor tells everyone what to count and collects the totals at the end. That's the **driver**.
- Counting inside one box is fast. The expensive part is when ballots must be **moved between officers**, for example to put all the votes from one constituency together. That's a **shuffle**, and it's the single most important idea in this chapter.
- If every officer gets a small box except one who gets a truckload, everyone waits for that one person. That's **skew**.
- If you only need votes from one district, you shouldn't open every box. That's **partition pruning**.
- If you only need the party name and not the whole ballot, you shouldn't read the whole ballot. That's **column pruning**.
- Hiring a thousand officers to count a village's votes is slower than one person doing it, because organizing them takes longer than the counting. That's Spark on small data.

---

## 48.0 Setting up Spark, Polars, and the data

This chapter uses two new tools. **Spark** is a distributed engine: it splits work across many machines, or, on your laptop, across your processor's cores. **PySpark** is the Python library you use to drive it. Spark itself runs on **Java**, so Java must be installed too. **Polars** is a DataFrame library like pandas, built for speed; section 48.6 introduces it.

Setting up takes one sitting. Java is the step most likely to cause trouble, so the troubleshooting box at the end of this section lists the usual errors.

### Step 1. Your practice folder

Work the way Chapter 17 did. In your file manager, copy the folder `companion/ch48` and paste it inside `work`, so you have `work/ch48` to practise in. It holds two files:

| File | What it does |
|---|---|
| `make_sensor_data.py` | Generates the sensor dataset (step 4). |
| `time_three_tools.py` | Times the same question in DuckDB, Polars, and Spark (section 48.6). |

### Step 2. Install Java

Spark 4.2 runs on Java 17, 21, or 25. Install **Java 21**, a long-term-support release, for your system:

- **Windows:** download the **Eclipse Temurin 21** installer (the `.msi` file) from adoptium.net. On the installer's *Custom Setup* page, turn on **Set JAVA_HOME variable**, then finish the install. `JAVA_HOME` is an environment variable (Chapter 34) that tells programs such as Spark where Java lives.
- **macOS:** download the **Eclipse Temurin 21** `.pkg` installer from adoptium.net and run it. (With Homebrew, `brew install --cask temurin@21` does the same.)
- **Linux, and WSL on Windows:** `sudo apt install openjdk-21-jre-headless` on Ubuntu or Debian. "Headless" means without the parts for desktop windows, which Spark doesn't need.

Open a **new** terminal (so it sees the new settings) and check:

**The chapter's output:**

```
# terminal
$ java -version
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-124.04, mixed mode, sharing)
```

**`java -version`** prints the version of the Java that your terminal finds. The first line is the one that matters: it must start with 17, 21, or 25. The build details on the next two lines depend on who made your Java; Temurin's say `Temurin`.

### Step 3. Install PySpark and Polars

In the terminal, go to your practice folder and activate the book's virtual environment (Chapter 17), then install:

**The chapter's output:**

```
# terminal
$ cd work/ch48
$ source ../../.venv/bin/activate
$ python -m pip install pyspark==4.2.0 polars==1.44.2
```

- **`source ../../.venv/bin/activate`** activates the environment, two levels up; on Windows PowerShell it's `..\..\.venv\Scripts\Activate.ps1`.
- **`pyspark`** is a large download (about 450 MB), because it carries Spark itself inside it. **`polars`** is Polars.
- `==4.2.0` and `==1.44.2` ask for the exact versions used for this chapter's outputs. The pins matter more than usual here: the text of Spark's plans changes a little between versions. `duckdb` is already installed from Chapter 45, and `pandas` and `pyarrow` from Chapter 18.

Add `pyspark` and `polars` to your `requirements.txt`, one per line (Chapter 26). Then check that both import:

**The chapter's output:**

```
# terminal
$ python -c "import pyspark, polars; print(pyspark.__version__, polars.__version__)"
4.2.0 1.44.2
```

### Step 4. Generate the data

**The chapter's output:**

```
# terminal
$ python make_sensor_data.py
sensor_readings: 19,872,000 readings, 114.3 MB
$ python make_sensor_data.py --small
sensor_readings_small: 216,000 readings, 1.2 MB
```

The first command builds the full dataset in the folder `sensor_readings`; it takes from a few seconds to a minute or two, depending on your computer. The second builds one day, `sensor_readings_small`, which section 48.6 uses for timing. What the generator makes:

- **25 machines**, `M-01` to `M-25`. Machines 1–15 are at **Bhiwandi Main** and 16–25 at **Chakan Pune**. Each machine has a **line type**, Injection, Extrusion, or Blow, taken in turn: Bhiwandi Main has 5 of each, and Chakan Pune has 3 Blow, 4 Extrusion, and 3 Injection.
- **One reading every 10 seconds**, 8,640 a day, from 1 October to 31 December 2025 (92 days). 25 × 8,640 × 92 = 19,872,000.
- Each reading has a **temperature** around a daily base between 180 and 212 °C, with random noise, and now and then a machine that drifts upwards through the day; a **pressure** that follows the temperature; **units made** (about 3 per reading); and a **scrap flag** that is true for about 1.2% of readings, and for 6% when the machine runs hot.
- The random numbers come from a fixed **seed** (48), so your data is identical to the book's, number for number.

The data is written as Parquet (Chapter 45), **partitioned by `reading_date`**: one folder per day, named in a special way. See for yourself. From here on, work in a Jupyter notebook: create `ch48.ipynb` in `work/ch48`, choose the `.venv` kernel (Chapter 17, section 17.0), and run each code block as its own cell, top to bottom, without restarting the kernel.

In [1]:
import os

days = sorted(os.listdir("sensor_readings"))
print(len(days), "folders, from", days[0], "to", days[-1])
print("inside the first:", os.listdir(os.path.join("sensor_readings", days[0])))

sizes = [os.path.getsize(os.path.join("sensor_readings", d, "part-0.parquet")) for d in days]
print(f"total {sum(sizes) / 1e6:.1f} MB; one day about {sizes[0] / 1e6:.1f} MB")

92 folders, from reading_date=2025-10-01 to reading_date=2025-12-31
inside the first: ['part-0.parquet']
total 114.3 MB; one day about 1.2 MB


- `os.listdir()` lists a folder's contents; `sorted()` puts the names in order.
- `os.path.join()` builds a path with the right separator for your system (`/` or `\`).
- The list comprehension (Chapter 17) collects each day's file size in bytes; dividing by `1e6` (one million) gives megabytes.

**Reading it.** Each folder is named **`reading_date=2025-10-01`**: the column name, an equals sign, and the value. This is **Hive-style partitioning** (named after the Hadoop tool that made it popular), and every engine in this chapter understands it. The files inside don't contain a `reading_date` column at all; the engines take it from the folder names. That's what lets them skip whole days they don't need (section 48.5).

Each day's folder is only about 1.2 MB. That's far below the file sizes this chapter recommends later (128–512 MB), and at this size daily folders are still fine, because there are only 92 files. With years of data you'd partition by month instead, or keep days and merge small files regularly (Chapter 49).

### Step 5. Your first Spark session

A Spark program starts by creating a **SparkSession**, the entry point to everything Spark does. This cell is also your test that Java and PySpark work together.

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("riverstone-sensors")
         .master("local[2]")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is running")

> **Not run here.** This cell needs something this machine does not have: a database, an API key, a running service, or command-line arguments. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
Spark 4.2.0 is running
```

**How it works, line by line.**

- The whole builder sits inside **round brackets** so that it can run over several lines, one setting per line. Python treats everything between the brackets as one expression.
- **`SparkSession.builder`** starts describing the session you want.
- **`.appName("riverstone-sensors")`** names the job. The name appears in Spark's web interface (section 48.5).
- **`.master("local[2]")`** says where the work runs: `local` means **local mode**, one process on this computer, and `[2]` means two worker threads, so two cores work in parallel. `local[*]` would use every core. This chapter uses two, so that your computer stays responsive and section 48.6 can give all three engines the same two cores. On a real cluster, this is where the cluster's address goes instead.
- **`.config("spark.ui.showConsoleProgress", "false")`** turns off the progress bar Spark would otherwise draw under each cell.
- **`.getOrCreate()`** creates the session, or, if one is already running, returns that one. That matters in Jupyter: re-running the cell doesn't start a second Spark.
- **`setLogLevel("ERROR")`** hides Spark's stream of information messages and warnings, so only real errors show.
- The first run takes several seconds, because Java starts up behind the scenes. Later cells are faster.

> **When the first Spark cell fails.**
>
> - **`JAVA_GATEWAY_EXITED`** or *"Java gateway process exited before sending its port number"*: Spark couldn't start Java. Java isn't installed, or `JAVA_HOME` isn't set. Check `java -version` in a *new* terminal, then restart VS Code so it sees the change.
> - **`UnsupportedClassVersionError`**: your Java is too old (Java 8 or 11). Install Java 21 and make sure `java -version` shows it.
> - A warning that **port 4040 is in use**: another Spark session is already running (perhaps in another notebook). Spark picks port 4041 instead. It's harmless.
> - On Windows, a warning about **`winutils.exe`** or **`HADOOP_HOME`**: harmless for reading data. Writing files with Spark on Windows (section 48.7) needs one extra download; the simplest route is to run this chapter in WSL (Chapter 34), or search for "Spark winutils HADOOP_HOME" for the Windows steps.
> - A **warning that PySpark "does not yet fully support pandas 3"**: nothing in this chapter depends on it. Ignore it.

> **Tool note.** The outputs in this chapter were produced with Python 3.11.15, PySpark 4.2.0, Polars 1.44.2, DuckDB 1.5.6, PyArrow 25.0.1, pandas 3.0.6, and OpenJDK 21.0.10. Newer versions may format a few outputs slightly differently.

---

## 48.1 Most "big data" isn't

A claim worth making early, because it has only become more true: **most datasets described as "big data" fit on one computer.**

A modern laptop has 8 to 32 GB of memory and 8 or more cores. Tools such as DuckDB and Polars are built to use them, to work directly on Parquet files, and to spill to disk when data doesn't fit in memory. In practice that means:

| Rough size | What usually works |
|---|---|
| Up to a few million rows | Spreadsheets struggle; SQL in a database, pandas, DuckDB, or Polars are all comfortable |
| Tens of millions of rows | DuckDB or Polars on a laptop; a warehouse if it's shared |
| Hundreds of millions to billions | A cloud warehouse (Chapter 49), or Spark, usually on a cluster |
| Continuous, never-ending data | Streaming (Chapter 50), which is a different problem |

Riverstone's entire order history is a few hundred thousand rows: a spreadsheet's worth. Its sensor data is far bigger, and this chapter's 19.87 million readings are three months from 25 machines. A year across more machines reaches hundreds of millions, which is where the question becomes real.

Before choosing a distributed tool, ask four questions.

1. **How much data does the job actually touch?** Not the size of the source; the size after filtering to the days and columns you need. Partitioned Parquet often turns "a billion rows" into "40 million".
2. **How often does it run, and how long can it take?** A nightly job with a four-hour window is a different problem from a query someone waits for.
3. **Does it fit in memory after the filter?** If yes, one machine is almost always faster and simpler.
4. **What's the cost of being wrong?** Starting simple and moving to Spark later is cheap. Starting on a cluster and discovering you never needed it costs months of complexity.

> **Watch out: "we might need it later" is not a reason.** Later is when to build it. A pipeline written in SQL or Polars can be rewritten in Spark in days, and most never need to be.

---

## 48.2 What "distributed" actually means

A distributed engine splits data into **partitions**, sends work on each partition to a **worker** (also called an executor), and coordinates everything from the **driver**, which is the process running your code.

Three consequences shape everything you write.

- **Work that stays inside a partition is cheap.** Filtering, selecting columns, arithmetic, and per-row work run in parallel with no coordination. These are **narrow** operations.
- **Work that needs data from other partitions is expensive.** Grouping, joining, sorting, and deduplication must first bring matching rows together, which means writing data to disk, sending it over the network, and reading it back. That movement is the **shuffle**, and these are **wide** operations.
- **Bringing results back to the driver is a bottleneck.** `collect()` pulls every row into the memory of one process. It's how many first Spark jobs die.

![A diagram of one Spark job. At the top, the driver, running your Python code and holding the plan. Below it, four workers, each holding two partitions of the sensor data. In each worker, a narrow stage, drawn with a solid border, where it filters its own partitions with no communication; then a wide stage, drawn with a dashed border, with arrows underneath running from each worker to its neighbours to bring all rows for the same machine together. An arrow from the workers up to the driver is labeled: collect() brings every row back into the driver's memory. A note underneath: narrow boxes are cheap and fully parallel; wide boxes are the cost, because rows cross between workers.](figures/fig48-1-driver-workers-shuffle.svg)

*Figure 48.1 — Driver, workers, partitions, and the shuffle. The crossing arrows are what you pay for.*

**Lazy execution** ties it together. Spark doesn't run anything as you build a query. It records what you asked for, plans it as a whole, and runs it when you ask for a result (an **action** such as `count()`, `show()`, `collect()`, or writing files). That's why Spark can rearrange your work, and why the plan is worth reading.

> **Simplification note.** Everything here runs Spark in **local mode**, one Java process on one machine, using your cores as workers. That's the standard way to develop and test, and every idea, plan, and optimization transfers to a cluster. What local mode can't show you is network cost and cluster tuning; section 48.6 is honest about what that means for the timings.

---

## 48.3 PySpark on the sensor data

Spark cuts the data into a number of pieces after every shuffle, set by `spark.sql.shuffle.partitions`. The default is 200, which is far too many for one machine and a common cause of slow local jobs. Set it once, then read the data:

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", 8)

readings = spark.read.parquet("sensor_readings")
print(f"rows: {readings.count():,}")
readings.printSchema()

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
rows: 19,872,000
root
 |-- machine_id: string (nullable = true)
 |-- plant: string (nullable = true)
 |-- line_type: string (nullable = true)
 |-- reading_ts: timestamp_ntz (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- pressure_bar: double (nullable = true)
 |-- units_made: integer (nullable = true)
 |-- scrap_flag: boolean (nullable = true)
 |-- reading_date: date (nullable = true)
```

- **`spark.conf.set(name, value)`** changes a setting on the running session. Eight pieces suits two cores; on a cluster, aim for a few times the total number of cores.
- **`spark.read.parquet("sensor_readings")`** reads the whole folder. Spark discovers the `reading_date` partition column from the folder names, which is why it appears in the schema without being in any file. The result, `readings`, is a Spark **DataFrame**: like a pandas DataFrame, but split into partitions and computed lazily.
- **`count()`** is an action: it runs a job and returns the number of rows.
- **`printSchema()`** prints each column's name and type. `timestamp_ntz` is a timestamp with **no time zone** ("ntz"); `nullable = true` means the column may hold missing values.

### A first real question

*"How many readings does each plant have?"* The simplest group-by:

In [ ]:
readings.groupBy("plant").count().orderBy("plant").show()

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
+-------------+--------+
|        plant|   count|
+-------------+--------+
|Bhiwandi Main|11923200|
|  Chakan Pune| 7948800|
+-------------+--------+
```

- **`groupBy("plant")`** groups rows by plant, as `GROUP BY` does in SQL and `groupby` in pandas.
- **`.count()`** after a `groupBy` counts rows per group, in a column called `count`.
- **`.orderBy("plant")`** sorts the result. Without it, groups can come back in any order.
- **`.show()`** is an action: it runs the job and prints up to 20 rows as a table.

Now the full question: *"What's the average temperature and scrap rate for each plant and line type?"*

In [ ]:
from pyspark.sql import functions as F

by_plant = (readings
            .groupBy("plant", "line_type")
            .agg(F.count("*").alias("readings"),
                 F.round(F.avg("temperature_c"), 2).alias("avg_temp_c"),
                 F.round(F.avg(F.col("scrap_flag").cast("int")) * 100, 2).alias("scrap_pct"))
            .orderBy("plant", "line_type"))
by_plant.show(truncate=False)

> **Not run here.** This cell needs something this machine does not have: a database, an API key, a running service, or command-line arguments. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
+-------------+---------+--------+----------+---------+
|plant        |line_type|readings|avg_temp_c|scrap_pct|
+-------------+---------+--------+----------+---------+
|Bhiwandi Main|Blow     |3974400 |196.34    |1.3      |
|Bhiwandi Main|Extrusion|3974400 |196.35    |1.29     |
|Bhiwandi Main|Injection|3974400 |196.35    |1.29     |
|Chakan Pune  |Blow     |2384640 |196.35    |1.3      |
|Chakan Pune  |Extrusion|3179520 |196.32    |1.3      |
|Chakan Pune  |Injection|2384640 |196.38    |1.29     |
+-------------+---------+--------+----------+---------+
```

**How it works, line by line.**

- **`from pyspark.sql import functions as F`** imports Spark's built-in functions under the short name `F`, the convention everyone uses. Don't write `from pyspark.sql.functions import *`: that would replace Python's own `sum`, `round`, and `max` with Spark's, and plain Python code later in the notebook would break in confusing ways.
- **`.agg(...)`** computes several results per group at once, one per argument.
- **`F.count("*")`** counts rows. **`.alias("readings")`** names the result column.
- **`F.avg("temperature_c")`** is the mean. **`F.round(x, 2)`** rounds it to two decimal places.
- **`F.col("scrap_flag")`** refers to a column. In most functions, `F.col("x")` and the plain string `"x"` mean the same thing; you need `F.col` when you do something to the column, like here.
- **`.cast("int")`** turns True and False into 1 and 0. The average of a 0/1 column is the **share** of ones: the values 1, 1, 1, 0 average 0.75, and three of the four are ones. Times 100 makes it a percentage.
- **`show(truncate=False)`** prints full values; by default `show()` cuts any value longer than 20 characters.

**Reading it.** Three line types at each plant, with scrap around 1.3% and average temperature near 196 °C. Bhiwandi Main has more readings because it has more machines (15 of the 25). This is a **wide** operation: the group-by needed a shuffle, which section 48.4 makes visible.

The PySpark DataFrame API will feel familiar after pandas (Chapter 18): `select`, `filter`, `groupBy`, `agg`, `join`, `orderBy`, and `withColumn` (which adds or replaces a column) all do what their names say. Spark also accepts SQL directly (`readings.createOrReplaceTempView("readings"); spark.sql("SELECT …")`), and the two are the same engine underneath. Use whichever makes the logic clearer.

### Lazy execution, seen

In [ ]:
import time

start = time.perf_counter()
hot = (readings.filter(F.col("temperature_c") > 200)
               .select("machine_id", "reading_ts", "temperature_c"))
build_seconds = time.perf_counter() - start

start = time.perf_counter()
hot_count = hot.count()                      # the action: this runs the job
count_seconds = time.perf_counter() - start

print(f"readings above 200 °C: {hot_count:,}")
print("counting took longer than building:", count_seconds > build_seconds)

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
readings above 200 °C: 8,143,256
counting took longer than building: True
```

- **`time.perf_counter()`** reads a precise clock, in seconds. The difference between two readings is how long the code between them took.
- **`filter`** keeps the rows where the condition is true; **`select`** keeps the named columns. The round brackets around `hot = (…)` let the chain run over two lines, as with the builder in section 48.0.

**Reading it.** Building `hot` touched no data: Spark only checked that the columns exist and wrote down the plan. `count()` is an action, so Spark planned and ran the job, scanning 19.87 million readings to find 8.14 million above 200 °C. On the book's test machine, building took between 0.03 and 0.16 seconds and counting between 0.6 and 1.3 seconds; yours will differ, which is why the cell prints a comparison rather than the times. On a cluster, the same code would run the same way across machines.

Four actions come up all the time: **`count()`**, **`show()`**, **`collect()`** (section 48.7), and **writing files** (also section 48.7). Everything else only adds to the plan.

---

## 48.4 Reading the plan, and finding the shuffle

A **query plan** is Spark's description of how it will do the work. Reading it is the difference between guessing at performance and knowing. Every DataFrame has an **`explain()`** method that prints its plan without running it.

In [ ]:
units = readings.groupBy("machine_id").agg(F.sum("units_made").alias("units"))

spark.conf.set("spark.sql.maxMetadataStringLength", 40)   # shorten long values, such as file paths
units.explain()

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[machine_id#0], functions=[sum(units_made#6)])
   +- Exchange hashpartitioning(machine_id#0, 8), ENSURE_REQUIREMENTS, [plan_id=234]
      +- HashAggregate(keys=[machine_id#0], functions=[partial_sum(units_made#6)])
         +- Project [machine_id#0, units_made#6]
            +- FileScan parquet [machine_id#0,units_made#6,reading_date#8] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<machine_id:string,units_made:int>
```

- **`F.sum("units_made")`** totals a column, per group.
- **`spark.sql.maxMetadataStringLength`** limits how long some values in the plan may be before Spark cuts them with `...`. The folder path is the long one: set to 40, your plan shows only the start of it, `[file:/home...` here, or something like `[file:/C:/Us...` on Windows.
- **`explain()`** prints the **physical plan**: the steps Spark will actually run.

The plan is a tree, printed upside down: each `+-` line is a step whose input is the line below it. Three kinds of noise make it look harder than it is:

- **`#0`, `#6`, `#8`** after each column name are Spark's internal ids for columns. They change from run to run, and yours may differ from the book's if you ran cells in a different order. Ignore them.
- **`[plan_id=…]`** is an internal number for the shuffle step, also different every run.
- The **`FileScan`** line is very long: it lists everything about the file read, separated by commas.

> **Try it.** Run `units.explain("formatted")`. The same plan comes out as a short numbered tree followed by one block of details per step, which is easier to read for long plans. `explain()` alone is what you'll see in most answers online and in interviews.

### Optional: a tidier printout

The rest of this chapter prints plans through a small helper, `plan()`, that removes the ids and puts each part of the `FileScan` line on its own line, so the plans fit the page and match what you'll see. It uses only `explain()` underneath. You can skip the helper and use `explain()` everywhere; the plans are the same.

In [ ]:
import io, re, contextlib

def plan(df, only=None):
    """Print df's physical plan without the ids, one FileScan detail per line."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):             # catch what explain() prints
        df.explain()
    text = re.sub(r"#\d+L?", "", buffer.getvalue())       # drop column ids like #6 or #56L
    text = re.sub(r", \[plan_id=\d+\]", "", text)         # drop plan ids
    for line in text.splitlines()[1:]:                   # skip "== Physical Plan =="
        parts = [line]
        if "FileScan" in line:                           # cut before each "Name: " field
            head, *fields = re.split(r", (?=[A-Za-z]+: )", line)
            parts = [head] + ["        " + f for f in fields
                              if f.startswith(("PartitionFilters", "PushedFilters", "ReadSchema"))]
        for part in parts:
            if part.strip() and (only is None or any(word in part for word in only)):
                print(part)

spark.conf.unset("spark.sql.maxMetadataStringLength")    # back to the default
plan(units)

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[machine_id], functions=[sum(units_made)])
   +- Exchange hashpartitioning(machine_id, 8), ENSURE_REQUIREMENTS
      +- HashAggregate(keys=[machine_id], functions=[partial_sum(units_made)])
         +- Project [machine_id, units_made]
            +- FileScan parquet [machine_id,units_made,reading_date] Batched: true
        PartitionFilters: []
        PushedFilters: []
        ReadSchema: struct<machine_id:string,units_made:int>
```

**How it works, line by line.**

- **`io.StringIO()`** is a text buffer that behaves like a file in memory. **`contextlib.redirect_stdout(buffer)`** sends everything printed inside the `with` block into the buffer instead of the screen, so `explain()`'s output ends up in `buffer.getvalue()` as a string.
- **`re.sub(pattern, replacement, text)`** replaces every match of a regular expression (Chapter 14, section 14.2's "Patterns in ten minutes"). `#\d+L?` means a `#`, one or more digits, and an optional `L`: the column ids. `, \[plan_id=\d+\]` matches the plan ids; the square brackets need a backslash because on their own they mean "one of these characters".
- **`text.splitlines()[1:]`** splits the text into lines and skips the first, the `== Physical Plan ==` title.
- **`re.split(r", (?=[A-Za-z]+: )", line)`** cuts the `FileScan` line at every comma-and-space that is followed by a word and a colon, such as `, ReadSchema: `. The `(?= … )` part means "followed by", without cutting that text off. Commas inside a list, such as `[IsNotNull(plant), EqualTo(…)]`, aren't followed by `Word: `, so the lists stay whole.
- **`head, *fields = …`** puts the first piece in `head` and all the others in the list `fields`.
- **`f.startswith((…, …, …))`** is true if `f` starts with any of the three strings in the tuple. The helper keeps only the three fields this chapter reads, indented by eight spaces.
- **`only`** is an optional list of words: when you pass it, only lines containing one of them are printed. `spark.conf.unset()` puts the setting from the previous cell back to its default.

### How to read it

From the bottom up, because that's the order things happen.

- **`FileScan parquet`** reads the files. `ReadSchema` shows Spark only reads two columns, `machine_id` and `units_made`. That's **column pruning**, free with Parquet. `reading_date` is listed in the brackets too, but it comes from the folder names, so it costs nothing to read.
- **`Project`** keeps only the columns needed.
- **`HashAggregate … partial_sum`** is a partial total computed inside each partition, with no communication. Doing this before the shuffle is why grouping is efficient: far less data has to move.
- **`Exchange hashpartitioning(machine_id, 8)`** is the **shuffle**: rows are redistributed so that every row with the same `machine_id` ends up in the same one of the 8 partitions. `ENSURE_REQUIREMENTS` means Spark added it because the next step needs it.
- **`HashAggregate`** at the top combines the partial totals into final ones.
- **`AdaptiveSparkPlan`** means adaptive query execution is on: Spark may change parts of the plan at runtime once it sees the real data sizes (section 48.5).

**Every `Exchange` is a shuffle, and every shuffle costs.** When a job is slow, count the exchanges first.

![Two plans, one above the other, each printed in full as the helper shows them. The first is the group-by plan from this section, with the Exchange hashpartitioning line highlighted and labeled: the shuffle, where rows move so that all readings for a machine sit together. The second is the pruning plan from section 48.5, with no Exchange, and with the PartitionFilters line, listing a single date, and the ReadSchema line highlighted and labeled: partition pruning and column pruning. Each highlighted line also has a marker symbol in the margin.](figures/fig48-2-reading-the-plan.svg)

*Figure 48.2 — The two things to look for in a plan: exchanges (cost) and pruning (work avoided).*

---

## 48.5 Making a job faster

### Partition pruning: don't open boxes you don't need

In [ ]:
december_1 = readings.filter((F.col("reading_date") == "2025-12-01") &
                             (F.col("plant") == "Bhiwandi Main"))
print(f"rows for 1 December at Bhiwandi Main: {december_1.count():,}")
plan(december_1.select("machine_id", "temperature_c"))

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
rows for 1 December at Bhiwandi Main: 129,600
*(1) Project [machine_id, temperature_c]
+- *(1) Filter (isnotnull(plant) AND (plant = Bhiwandi Main))
   +- *(1) ColumnarToRow
      +- FileScan parquet [machine_id,plant,temperature_c,reading_date] Batched: true
        PartitionFilters: [isnotnull(reading_date), (reading_date = 2025-12-01)]
        PushedFilters: [IsNotNull(plant), EqualTo(plant,Bhiwandi Main)]
        ReadSchema: struct<machine_id:string,plant:string,temperature_c:double>
```

- **`&`** combines two conditions: both must be true, as `AND` does in SQL. Each condition needs its own round brackets, exactly as in pandas (Chapter 18).

**Reading it.** The plan has **no Exchange at all** (this query needs no shuffle), and two lines matter:

- **`PartitionFilters: [isnotnull(reading_date), (reading_date = 2025-12-01)]`** means Spark reads one day's folder out of 92. It never opens the other 91.
- **`PushedFilters: [IsNotNull(plant), EqualTo(plant,Bhiwandi Main)]`** means the plant filter is handed to the Parquet reader, which can skip chunks of a file (row groups) whose statistics rule the value out. That only helps when the values are clustered, for example if the file were sorted by plant (Chapter 49). Here each day's file mixes both plants, so the filter is applied as the rows are read.

Between them, a query over 19.87 million readings touched 216,000. **Partitioning your storage by the column people filter on** (usually a date) is the single biggest performance decision in a data platform.

Pruning works only when Spark can see that a filter is about the partition column. Compare two filters that you know mean nearly the same thing:

In [ ]:
plan(readings.filter(F.month("reading_date") == 12).select("machine_id"), only=["PartitionFilters"])
plan(readings.filter(F.year("reading_ts") == 2025).select("machine_id"), only=["PartitionFilters"])

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
        PartitionFilters: [isnotnull(reading_date), (month(reading_date) = 12)]
        PartitionFilters: []
```

- **`F.month()`** and **`F.year()`** take the month and year out of a date or timestamp.
- **`only=["PartitionFilters"]`** prints just that line of each plan.

**Reading it.** An expression on the partition column itself, `month(reading_date) = 12`, still prunes: Spark checks it against each folder name. A filter on `reading_ts` doesn't, even though you know `reading_ts` and `reading_date` always agree. Spark doesn't know that, so `PartitionFilters` is empty and it opens every folder.

> **Watch out: a filter Spark can't see doesn't prune.** Filtering with a Python function (a user-defined function, section 48.7), or on a different column that carries the same information (`F.year("reading_ts") == 2025`), hides the filter from the planner, and Spark reads everything. Filter on the partition column, and check `PartitionFilters` in the plan.

### Joins: the shuffle you can often avoid

Suppose the plant team wants readings per supervisor. The supervisors live in a tiny table. `createDataFrame` makes a small Spark DataFrame from Python data:

In [ ]:
machines = spark.createDataFrame(
    [("M-01", "Anil Deshmukh", 2019), ("M-02", "Sunita Rao", 2021), ("M-03", "Imran Shaikh", 2017)],
    ["machine_id", "supervisor", "installed_year"])
machines.show()

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
+----------+-------------+--------------+
|machine_id|   supervisor|installed_year|
+----------+-------------+--------------+
|      M-01|Anil Deshmukh|          2019|
|      M-02|   Sunita Rao|          2021|
|      M-03| Imran Shaikh|          2017|
+----------+-------------+--------------+
```

- The first argument is a **list of tuples** (Chapter 17), one tuple per row; the second is the list of column names, in the same order.

Joining the readings to this table needs matching rows together, which normally means shuffling **both** sides. Spark would normally notice that `machines` is tiny and avoid that (as you'll see next), so the first line switches that off, to show the expensive plan:

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)   # don't broadcast automatically
joined = (readings.join(machines, on="machine_id", how="inner")
                  .groupBy("supervisor").agg(F.count("*").alias("readings")))
plan(joined, only=["Join", "Exchange"])
joined.orderBy("supervisor").show()

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
   +- Exchange hashpartitioning(supervisor, 8), ENSURE_REQUIREMENTS
            +- SortMergeJoin [machine_id], [machine_id], Inner
               :  +- Exchange hashpartitioning(machine_id, 8), ENSURE_REQUIREMENTS
                  +- Exchange hashpartitioning(machine_id, 8), ENSURE_REQUIREMENTS
+-------------+--------+
|   supervisor|readings|
+-------------+--------+
|Anil Deshmukh|  794880|
| Imran Shaikh|  794880|
|   Sunita Rao|  794880|
+-------------+--------+
```

- **`join(other, on="machine_id", how="inner")`** joins on the `machine_id` column. `how="inner"` keeps only readings whose machine is in both tables, as `INNER JOIN` does (Chapter 12). That's why only 3 of the 25 machines appear in the result.
- **`only=["Join", "Exchange"]`** prints only the join and shuffle lines of a longer plan.

**Reading it.** `SortMergeJoin` with two exchanges below it: both sides are redistributed by `machine_id`. That means 19.87 million rows are written to shuffle files (and, on a cluster, sent across the network) to be matched with three rows. The results are right (794,880 readings for each machine: 92 days × 8,640), and the work is wasteful.

The fix is a **broadcast join**: send the small table to every worker, so the big table never moves.

In [ ]:
broadcast_join = (readings.join(F.broadcast(machines), on="machine_id")
                          .groupBy("supervisor").agg(F.count("*").alias("readings")))
plan(broadcast_join, only=["Join", "Exchange"])
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)   # back to the default

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
   +- Exchange hashpartitioning(supervisor, 8), ENSURE_REQUIREMENTS
            +- BroadcastHashJoin [machine_id], [machine_id], Inner, BuildRight, false, false
               +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false)
```

- **`F.broadcast(machines)`** marks the small table to be copied to every worker. `how` is left out, and defaults to `"inner"`.
- The last line puts the automatic threshold back to its default, 10 MB (10 × 1,024 × 1,024 bytes).

**Reading it.** `BroadcastHashJoin` with a `BroadcastExchange` for the small side, and **no exchange on the big side**. The big data stays where it is. Spark does this automatically when it can tell a table is small (`spark.sql.autoBroadcastJoinThreshold`, 10 MB by default); `F.broadcast()` tells it explicitly, which matters when the size estimate is wrong, for example after a complex sub-query.

The remaining `Exchange hashpartitioning(supervisor, 8)` is the group-by, which still needs a shuffle. One shuffle instead of three.

### Skew: when one worker gets a truckload

**Skew** is uneven partitions. If half the readings come from one machine, the worker holding that machine's rows does half the work while others idle. Symptoms: a stage where 199 tasks finish in seconds and one runs for an hour, and memory errors on a single task.

The place to see skew is **Spark's web interface**. While a session is running, open `http://localhost:4040` in your browser. The *Stages* tab lists each stage of your jobs; click one and compare the **maximum** task duration with the **median**. A maximum many times the median is skew.

Usual fixes:

- **Adaptive query execution** (AQE, on by default since Spark 3.2) detects skewed join partitions and splits them. Often this is enough.
- **Salting:** add a random number to the join key on the big side and expand the small side to match, spreading the hot key across workers.
- **Handle the hot key separately:** filter it out, process it on its own, and union the results.
- **Filter earlier**, so the skewed rows never reach the join.

Salting is for joins where **both** sides are large. With a three-row table you'd simply broadcast, as above; the example below uses `machines` only to keep it small enough to see. First the small side: every machine row is copied once per salt value.

In [ ]:
SALTS = 8
small = machines.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(SALTS)])))
print("small side:", small.count(), "rows")
small.filter(F.col("machine_id") == "M-01").select("machine_id", "salt").show()

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
small side: 24 rows
+----------+----+
|machine_id|salt|
+----------+----+
|      M-01|   0|
|      M-01|   1|
|      M-01|   2|
|      M-01|   3|
|      M-01|   4|
|      M-01|   5|
|      M-01|   6|
|      M-01|   7|
+----------+----+
```

- **`withColumn(name, value)`** adds a column (or replaces one with the same name).
- **`F.lit(i)`** turns a plain Python value into a Spark column holding that value in every row. The list comprehension makes eight of them, for 0 to 7.
- **`F.array([...])`** puts those eight values into one array (a list) per row, and **`F.explode()`** turns each element of an array into its own row. So each machine row becomes eight rows, with salt 0 to 7: 3 × 8 = 24.

Then the big side: each reading gets one random salt, and the join uses both columns.

In [ ]:
big = readings.withColumn("salt", (F.rand(seed=42) * SALTS).cast("int"))
big.groupBy("salt").count().orderBy("salt").show()

salted = big.join(small, on=["machine_id", "salt"])
plain = readings.join(machines, on="machine_id")
print(f"salted join: {salted.count():,} rows; plain join: {plain.count():,} rows")

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
+----+-------+
|salt|  count|
+----+-------+
|   0|2485538|
|   1|2486457|
|   2|2481887|
|   3|2483628|
|   4|2483236|
|   5|2483089|
|   6|2482425|
|   7|2485740|
+----+-------+

salted join: 2,384,640 rows; plain join: 2,384,640 rows
```

- **`F.rand(seed=42)`** gives each row a random number between 0 and 1. The **seed** makes the numbers repeatable, so your counts match the book's.
- Times `SALTS` and **`.cast("int")`**, which drops the decimal part, gives a whole number from 0 to 7.
- **`on=["machine_id", "salt"]`** joins on both columns: a reading with salt 5 meets only the copy of its machine with salt 5.

**Reading it.** The salts spread the readings evenly, about 2.48 million each, so a hot machine's rows would now land in eight partitions instead of one. The last line is the check that matters: the salted join returns exactly as many rows as the plain one, 2,384,640 (3 machines × 794,880). Salting changed how the work is split, not the answer.

### A short list of what actually helps

| Change | Why it helps | Look for in the plan |
|---|---|---|
| Partition storage by date | Whole folders are skipped | `PartitionFilters` |
| Select only needed columns | Less data read and moved | `ReadSchema` |
| Filter before joining and grouping | Less data shuffled | Filters below the Exchange |
| Broadcast small tables | The big side never moves | `BroadcastHashJoin`, no Exchange on the big side |
| Set `shuffle.partitions` sensibly | Avoids thousands of tiny tasks | Number in `hashpartitioning(…, n)` |
| Write files of a sensible size (128–512 MB, once the data is that large) | Fewer, bigger reads | Number of output files |
| Cache only what's reused several times | Avoids recomputation; costs memory | `InMemoryTableScan` |

---

## 48.6 The same question in three tools

Now the comparison that matters in practice: one question, three engines. Two of them you've met, DuckDB (Chapter 45) and Spark. The third, Polars, needs a page of its own first.

### Polars in one page

**Polars** is a DataFrame library, like pandas, written for speed: it uses every core it's allowed and can plan a whole query before running it, as Spark does. To compare the engines fairly, this chapter gives Polars the same two cores as Spark. That setting must come before Polars is first imported:

In [16]:
os.environ["POLARS_MAX_THREADS"] = "2"
import polars as pl

day = pl.read_parquet("sensor_readings/reading_date=2025-12-01/part-0.parquet")
print(day.shape)
print(day.select("machine_id", "reading_ts", "temperature_c", "units_made", "scrap_flag").head(3))

(216000, 8)
shape: (3, 5)
┌────────────┬─────────────────────┬───────────────┬────────────┬────────────┐
│ machine_id ┆ reading_ts          ┆ temperature_c ┆ units_made ┆ scrap_flag │
│ ---        ┆ ---                 ┆ ---           ┆ ---        ┆ ---        │
│ str        ┆ datetime[ms]        ┆ f64           ┆ i32        ┆ bool       │
╞════════════╪═════════════════════╪═══════════════╪════════════╪════════════╡
│ M-01       ┆ 2025-12-01 00:00:00 ┆ 187.28        ┆ 5          ┆ false      │
│ M-01       ┆ 2025-12-01 00:00:10 ┆ 187.11        ┆ 2          ┆ false      │
│ M-01       ┆ 2025-12-01 00:00:20 ┆ 185.26        ┆ 2          ┆ false      │
└────────────┴─────────────────────┴───────────────┴────────────┴────────────┘


- **`os.environ[...] = "2"`** sets an environment variable for this notebook's process; Polars reads `POLARS_MAX_THREADS` once, when it's imported.
- **`import polars as pl`** is the usual short name.
- **`pl.read_parquet()`** reads one file, straight away, into a Polars DataFrame. **`.shape`** is (rows, columns), as in pandas. There is no `reading_date` column: this is one file, read without its folder name.
- **`select("machine_id", …)`** with column names keeps those five columns, so the table fits the page, and **`head(3)`** shows the first three rows. Polars prints each column's **type** under its name: `str` is text, `datetime[ms]` a timestamp stored to the millisecond, `f64` a decimal number, `i32` a 32-bit whole number, and `bool` True/False.

Polars works through **expressions**: small recipes such as "the `temperature_c` column, averaged", which you hand to a method like `select` or `filter`:

In [17]:
print(day.select(pl.col("temperature_c").mean().round(2)))
print(day.filter(pl.col("temperature_c") > 200).height, "readings above 200 °C that day")

shape: (1, 1)
┌───────────────┐
│ temperature_c │
│ ---           │
│ f64           │
╞═══════════════╡
│ 196.49        │
└───────────────┘
86879 readings above 200 °C that day


- **`pl.col("temperature_c")`** means "the `temperature_c` column", like Spark's `F.col`. Methods chain onto it: `.mean()`, then `.round(2)`.
- **`select(...)`** returns a new DataFrame with the expressions' results. **`filter(...)`** keeps the rows where the expression is true; **`.height`** is the number of rows.

Why expressions, instead of pandas-style `day["temperature_c"].mean()`? Because an expression describes the work without doing it, Polars can see a whole query at once and plan it, as Spark does. That's what **lazy** mode uses. **`pl.scan_parquet()`** doesn't read anything; it starts a plan. **`.collect()`** runs it, and **`.explain()`** shows the optimized plan first:

In [18]:
lazy = (pl.scan_parquet("sensor_readings/*/*.parquet")
          .filter(pl.col("temperature_c") > 200)
          .select(pl.len()))
print(lazy.explain())
print(lazy.collect())

SELECT [len()]
  simple π 0/0 []
    Parquet SCAN [sensor_readings/reading_date=2025-10-01/part-0.parquet, ... 91 other sources]
    PROJECT 1/8 COLUMNS
    SELECTION: col("temperature_c") > 200.0
    ESTIMATED ROWS: 19872614
shape: (1, 1)
┌─────────┐
│ len     │
│ ---     │
│ u32     │
╞═════════╡
│ 8143256 │
└─────────┘


- **`"sensor_readings/*/*.parquet"`** is a pattern: every Parquet file in every day folder (`*` matches any name, as in the terminal, Chapter 34).
- **`pl.len()`** counts rows.

**Reading it.** Read it from the bottom up, as in Spark. The `Parquet SCAN` covers all 92 files, with a `SELECTION` (the filter, pushed into the read) and `PROJECT 1/8 COLUMNS`: of the eight columns in each file, only `temperature_c` is read. That's column pruning and a pushed filter, the same ideas you saw in Spark's plan. `ESTIMATED ROWS` is Polars' guess before reading, not a count. The count itself, 8,143,256, matches Spark's.

### One question, three engines

*"For readings above 190 °C, what's the average temperature, scrap rate, and total output per plant?"* First DuckDB, in SQL:

In [19]:
import duckdb

QUESTION = """
    SELECT plant,
           ROUND(AVG(temperature_c), 2)                 AS avg_temp_c,
           ROUND(AVG(CAST(scrap_flag AS INT)) * 100, 2) AS scrap_pct,
           SUM(units_made)::BIGINT                      AS units
    FROM readings
    WHERE temperature_c > 190
    GROUP BY plant
    ORDER BY plant"""

con = duckdb.connect()
con.execute("SET threads = 2")
con.execute("CREATE VIEW readings AS SELECT * FROM read_parquet('sensor_readings/*/*.parquet')")
print(con.execute(QUESTION).df().to_string(index=False))

        plant  avg_temp_c  scrap_pct    units
Bhiwandi Main      203.43       1.33 23234149
  Chakan Pune      203.44       1.33 15491402


- **`duckdb.connect()`** with no file name opens a database in memory. **`SET threads = 2`** gives it two cores, like the other two engines.
- **`read_parquet('sensor_readings/*/*.parquet')`** reads every file; DuckDB recognizes the `reading_date=` folder names and adds `reading_date` as a column automatically. The **view** lets the question say `FROM readings`.
- **`.df()`** returns the result as a pandas DataFrame (Chapter 18), and **`to_string(index=False)`** prints it without the row numbers.

Then Polars, one method per line:

In [20]:
polars_answer = (pl.scan_parquet("sensor_readings/*/*.parquet")
                   .filter(pl.col("temperature_c") > 190)
                   .group_by("plant")
                   .agg(pl.col("temperature_c").mean().round(2).alias("avg_temp_c"),
                        (pl.col("scrap_flag").cast(pl.Int8).mean() * 100).round(2).alias("scrap_pct"),
                        pl.col("units_made").sum().alias("units"))
                   .sort("plant")
                   .collect())
print(polars_answer)

shape: (2, 4)
┌───────────────┬────────────┬───────────┬──────────┐
│ plant         ┆ avg_temp_c ┆ scrap_pct ┆ units    │
│ ---           ┆ ---        ┆ ---       ┆ ---      │
│ str           ┆ f64        ┆ f64       ┆ i32      │
╞═══════════════╪════════════╪═══════════╪══════════╡
│ Bhiwandi Main ┆ 203.43     ┆ 1.33      ┆ 23234149 │
│ Chakan Pune   ┆ 203.44     ┆ 1.33      ┆ 15491402 │
└───────────────┴────────────┴───────────┴──────────┘


- **`group_by("plant")`** then **`agg(...)`** computes one result per expression for each plant, like Spark's `groupBy().agg()`. **`.alias()`** names each result.
- **`.cast(pl.Int8)`** turns True/False into 1/0 (as a small 8-bit whole number), so that `.mean()` gives the share of scrap readings, the same trick as in Spark.
- **`.sort("plant")`** orders the rows; **`.collect()`** runs the whole plan.
- Polars keeps the input's type for the sum, 32-bit whole numbers (`i32`), which hold values up to about 2.1 billion. The stretch goal's ten-times-larger data is still safe; for bigger totals, add `.cast(pl.Int64)` before `.sum()`.

And Spark, with the functions from section 48.3:

In [ ]:
(readings.filter(F.col("temperature_c") > 190)
         .groupBy("plant")
         .agg(F.round(F.avg("temperature_c"), 2).alias("avg_temp_c"),
              F.round(F.avg(F.col("scrap_flag").cast("int")) * 100, 2).alias("scrap_pct"),
              F.sum("units_made").alias("units"))
         .orderBy("plant")
         .show())

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
+-------------+----------+---------+--------+
|        plant|avg_temp_c|scrap_pct|   units|
+-------------+----------+---------+--------+
|Bhiwandi Main|    203.43|     1.33|23234149|
|  Chakan Pune|    203.44|     1.33|15491402|
+-------------+----------+---------+--------+
```

**Reading it.** All three agree to the last unit: 203.43 °C and 1.33% scrap at Bhiwandi Main, 23,234,149 units. That's the first thing to check when comparing tools, and it's reassuring how often people skip it.

### Timings

Speed depends on the machine, so these numbers are **measurements from one specific setup, not a benchmark**. The timing code is short. It runs each engine three times and keeps the fastest run:

In [ ]:
def best_of_3(job):
    """Run job three times; return the fastest time, in milliseconds."""
    times = []
    for _ in range(3):
        start = time.perf_counter()
        job()
        times.append(time.perf_counter() - start)
    return min(times) * 1000

duckdb_job = lambda: con.execute(QUESTION).fetchall()
polars_job = lambda: (pl.scan_parquet("sensor_readings/*/*.parquet")
                        .filter(pl.col("temperature_c") > 190).group_by("plant")
                        .agg(pl.col("temperature_c").mean(), pl.col("scrap_flag").cast(pl.Int8).mean(),
                             pl.col("units_made").sum())
                        .collect())
spark_job = lambda: (readings.filter(F.col("temperature_c") > 190).groupBy("plant")
                        .agg(F.avg("temperature_c"), F.avg(F.col("scrap_flag").cast("int")),
                             F.sum("units_made"))
                        .collect())

for name, job in [("DuckDB", duckdb_job), ("Polars", polars_job), ("Spark", spark_job)]:
    print(f"{name:<7}{best_of_3(job):8.1f} ms")

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

> **Not run here.** The chapter marks this block as illustration rather than something to execute, usually because it needs a service, a key or a file this notebook does not set up. Read it, do not run it.

**The chapter's output:**

```
DuckDB    231.7 ms
Polars    273.2 ms
Spark    1137.0 ms
```

**How it works, line by line.**

- **`best_of_3(job)`** takes a **function** as its argument and calls it three times, timing each call with `time.perf_counter()`. `for _ in range(3)` repeats three times; `_` is the usual name for a loop variable you don't use.
- It returns the **fastest** of the three. The first run is usually the slowest, because the files aren't yet in the operating system's memory cache and, for Spark, Java is still warming up. The fastest run measures the engine, not the disk.
- **`lambda: …`** (Chapter 18, "Functions without names") makes a small function with no arguments. `duckdb_job` doesn't run the query; calling `duckdb_job()` does, which is what `best_of_3` needs.
- Each job is the whole question, including bringing the two result rows back to Python: `fetchall()` for DuckDB, `collect()` for Polars and Spark. The Spark job reads through the `readings` DataFrame from section 48.3, so Spark's start-up time isn't counted.
- The `for` loop takes each (name, function) pair in turn. `{name:<7}` pads the name to 7 characters so the numbers line up, and `:8.1f` prints the time 8 characters wide with one decimal place.

Your numbers will differ, and should: they depend on your processor, your disk, and what else is running. The companion script `time_three_tools.py` runs the same timings on both the one-day folder and the full data. On the book's test machine, a shared computer with 4 cores, with each engine given two of them, it measured:

| Data | DuckDB | Polars | Spark (local) |
|---|--:|--:|--:|
| 216,000 readings (one day) | 6.8 ms | 5.8 ms | 252.3 ms |
| 19,872,000 readings (92 days) | 213.0 ms | 262.3 ms | 1,211.8 ms |

**What this does and doesn't show.**

- On a single machine, at this size, **DuckDB and Polars are several times faster than Spark**, and far simpler to run. On the small file, Spark's overhead dominates completely.
- The gap is not a flaw in Spark. Spark is paying for machinery that lets the same code run over hundreds of machines, and here it has only two cores, so its parallelism has almost nowhere to go.
- What these numbers *can't* show is the crossover: a dataset large enough that one machine runs out of memory or time, where Spark keeps going by adding machines. You'll reach that point with hundreds of millions to billions of rows, or with jobs that need more memory than a machine can hold.

> **Try it.** Run `python time_three_tools.py` in your practice folder and compare. On 8 cores, all three get faster: DuckDB and Polars use every core too. Spark's fixed start-up cost stays, so on the one-day file it will still be slowest by far. The gap on the full data may narrow, but on one machine it rarely reverses. Spark's advantage appears only when the data or the work outgrows one machine.

### Choosing

| Choose | When |
|---|---|
| **SQL in your warehouse** | The data is already there and the query is a query (most of the time) |
| **DuckDB** | Files on disk or in cloud storage, SQL, one machine, up to tens of millions of rows and beyond |
| **Polars** | The same, expressed as Python DataFrame code, or inside an existing Python pipeline |
| **Spark** | Data too large for one machine, jobs that must scale with the business, streaming at volume, or a platform (Databricks, EMR) your company already runs |
| **A cluster of any kind** | Only when a single machine has been tried and can't do it |

---

## 48.7 Writing Spark that doesn't surprise you

**Don't `collect()` big results.** `collect()` is the action that brings rows back into Python, as a list, and it pulls every one of them into the driver's memory. Bring back aggregates, samples, or filtered slices:

In [ ]:
one_machine_day = readings.filter((F.col("reading_date") == "2025-12-01") &
                                  (F.col("machine_id") == "M-07"))
rows = one_machine_day.select("reading_ts", "temperature_c").limit(3).collect()
print("three rows brought back to Python:", [(str(r[0]), r[1]) for r in rows])
print(f"rows that would come back without the filter: {readings.count():,}")

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
three rows brought back to Python: [('2025-12-01 00:00:00', 195.78), ('2025-12-01 00:00:10', 191.9), ('2025-12-01 00:00:20', 195.91)]
rows that would come back without the filter: 19,872,000
```

- **`limit(3)`** keeps only the first three rows, before anything leaves Spark.
- `collect()` returns a list of **`Row`** objects. A `Row` works like a tuple: `r[0]` is its first value (the timestamp) and `r[1]` its second. `str()` turns the timestamp into readable text.

**Write full results to files, and watch how many.** Spark writes one file per partition. Straight after a shuffle that can be up to `spark.sql.shuffle.partitions` files: 200 by default. Adaptive execution often merges small partitions, but don't rely on it for file sizes. Thousands of small files make every later read slow (the "small files problem", Chapter 49). This writes a small weekly summary as one file:

In [ ]:
weekly = (readings.groupBy("plant", F.weekofyear("reading_date").alias("week"))
                  .agg(F.sum("units_made").alias("units"),
                       F.round(F.avg(F.col("scrap_flag").cast("int")) * 100, 2).alias("scrap_pct")))
weekly.coalesce(1).write.mode("overwrite").parquet("out/weekly_scrap")

files = sorted(f for f in os.listdir("out/weekly_scrap") if not f.startswith("."))
print(len(files), "files:", [f if f == "_SUCCESS" else f[:10] + "…" for f in files])
print("rows written:", spark.read.parquet("out/weekly_scrap").count())

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
2 files: ['_SUCCESS', 'part-00000…']
rows written: 28
```

- **`F.weekofyear()`** gives the week number of a date, so the result has one row per plant per week. The `agg(...)` with `F.sum`, `F.avg`, `.cast("int")`, `F.round`, and `.alias` works exactly as in section 48.3.
- **`coalesce(1)`** merges the partitions into one without a shuffle, so one file is written. **`repartition(n)`** does a full shuffle into exactly `n` evenly sized partitions; use it when you need more files, or balanced ones.
- **`.write.mode("overwrite").parquet(path)`** writes the result as Parquet. `"overwrite"` replaces whatever is there, so running the cell again gives the same result: the idempotent re-run of Chapter 46. The default mode, `"error"`, refuses to write if the folder exists.
- `sorted(f for f in … if not f.startswith("."))` lists the output folder, leaving out hidden files. In the `print`, `f if f == "_SUCCESS" else f[:10] + "…"` is Python's one-line *if/else*: it keeps `_SUCCESS` whole and shortens the long data-file name to its first 10 characters.
- Spark writes a **folder**, not a file. The listing skips hidden files (names starting with a dot, small checksum files Spark adds). What's left is one data file, whose full name looks like `part-00000-` followed by a long random id and `.snappy.parquet`, and an empty file called **`_SUCCESS`**, which Spark writes last so that a later job can tell the output is complete.
- For output partitioned into folders, as the sensor archive is, add **`.partitionBy("week")`** before `.parquet()`.

Aim for files of roughly 128–512 MB **once your data is that large**. This summary is a few kilobytes, so one file is right.

The rest of the rules:

- **Cache deliberately.** `df.cache()` keeps a DataFrame in memory for reuse. It helps when the same intermediate result is used several times; otherwise it wastes memory you'd rather give to the shuffle.
- **Prefer built-in functions over Python UDFs.** A Python **user-defined function** (UDF) is your own Python function applied to every row. It moves every row out of Java into Python and back, and blocks the optimizer from pushing filters down. Built-ins in `pyspark.sql.functions` stay in the engine.
- **Test on a small partition.** Run your job on one day before running it on a year: same code, one folder.
- **Keep work off the driver, and keep the job idempotent.** Bring back only small results, and don't loop over rows in Python. Chapter 46's rules apply: writes replace partitions, and a re-run is safe.

### The settings in this chapter

| Setting | What it does | What happens if you change it |
|---|---|---|
| `.master("local[2]")` | Runs Spark on this computer with two worker threads | `local[*]` uses every core: faster, but the rest of the computer slows down while a job runs |
| `spark.sql.shuffle.partitions` (set to 8) | How many pieces the data is cut into after each shuffle | Left at 200 on a laptop: hundreds of tiny tasks and slower jobs. Too few on a cluster: most cores sit idle |
| `spark.sql.autoBroadcastJoinThreshold` (10 MB) | Tables smaller than this are broadcast automatically | `-1` switches automatic broadcasting off (section 48.5). Much larger: Spark may copy a big table to every worker and run out of memory |
| `spark.sql.maxMetadataStringLength` (100) | How long a value in a printed plan may be before it's cut | Smaller hides detail, such as long filter lists; it changes only the printout, never the job |
| `spark.ui.showConsoleProgress` | Draws a progress bar under each cell | `"true"` brings the bar back: handy for long jobs, noisy in a notebook |

---

## 48.8 The wider landscape in one page

| Thing | What it is | Where you'll meet it |
|---|---|---|
| **Apache Spark** | The distributed engine in this chapter | Almost everywhere; the default for large batch work |
| **Databricks** | A managed platform built around Spark, with notebooks, Delta Lake, and job scheduling | Large enterprises; very common in job descriptions |
| **Amazon EMR, Google Dataproc, Azure Synapse/Fabric** | Cloud-managed Spark (and friends) | Companies standardized on one cloud |
| **Delta Lake, Apache Iceberg, Apache Hudi** | Table formats that add transactions and time travel to files | Chapter 49 |
| **Trino / Presto** | A query engine that runs SQL across many sources without moving the data | Ad-hoc analytics over a lake |
| **Ray, Dask** | Distributed Python for machine learning and general workloads | Data science teams scaling Python |
| **Hadoop and MapReduce** | The original distributed platform; HDFS still appears | Older systems; mostly historical now |
| **Snowflake, BigQuery, Redshift** | Cloud warehouses that scale SQL without you managing machines | Chapter 49; often the simpler answer |

The honest summary for a career: learn **SQL and one single-machine engine** deeply, learn **Spark's model** well enough to read plans and write correct jobs, and learn a **cloud platform** when a job needs one. Interviews for data engineering roles ask about Spark far more often than most day jobs use it, which is a reason to understand it, not a reason to build on it.

When you're done for the day, `spark.stop()` shuts the session down and frees its memory.

---

## Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Java missing or too old | `JAVA_GATEWAY_EXITED` or `UnsupportedClassVersionError` on the first cell | Install Java 21; check `java -version` (48.0) |
| Reaching for Spark on small data | A five-second job takes two minutes | Measure first; DuckDB or Polars (48.1, 48.6) |
| Leaving `shuffle.partitions` at 200 locally | Hundreds of tiny tasks, slow jobs | Set it to a few times your core count |
| Not looking at the plan | Guessing at why a job is slow | `explain()`; count the exchanges (48.4) |
| Filtering on `reading_ts` instead of the partition column | `PartitionFilters` empty; whole dataset read | Filter on the partition column (48.5) |
| Shuffling a big table to join a tiny one | `SortMergeJoin` with two exchanges | Broadcast the small side (48.5) |
| Ignoring skew | One task runs for hours; memory errors | AQE, salting, or handle the hot key separately |
| `collect()` on a large DataFrame | Driver runs out of memory | Aggregate, sample, or write to files (48.7) |
| Python UDFs in hot paths | Slow stages; optimizer can't push filters | Use built-in functions |
| Caching everything | Memory pressure, slower shuffles | Cache only what's reused |
| Thousands of tiny output files | Later reads get slower every week | `coalesce()` or `repartition()` before writing (48.7) |
| Comparing tools without comparing answers | A "fast" query that's wrong | Check the numbers match first (48.6) |
| Treating one machine's benchmark as universal | Wrong tool chosen for a bigger job | State the hardware; know where the crossover is |

---

## In the real world: the cluster that was slower than the laptop

Riverstone's plant team wanted a weekly report: for each machine, how long it ran above its safe temperature, and how that lined up with scrap. By then sensors covered about 200 machines across both plants, and the archive held six months of readings: about 300 million.

The contractor who took the job had done large Spark work before, and quoted a managed Spark cluster. Within three weeks there was a cluster, a notebook, and a job that took 26 minutes and cost more per month than the warehouse.

Meera asked the one question that changed the project: *"How much data does the report actually read?"*

The answer was a week at a time: about 12 million readings, filtered to one condition, grouped by machine. The cluster was reading the whole archive every run, because the files were partitioned by machine rather than by date, and the report's date filter couldn't prune anything.

Two changes followed. The archive was rewritten partitioned by date, with machine as a sorted column inside each day's files. Then the same report was written in DuckDB, reading the week's folders directly from cloud storage, and it finished in under a minute on a small virtual machine.

The cluster wasn't wrong in principle. It was solving a problem the storage layout had created. Riverstone kept Spark for one genuinely large job, the yearly rebuild across the full archive, and ran everything else on a single machine.

At the review, the contractor made the point that mattered most. "The 26 minutes weren't Spark's fault," he said. "We were reading 300 million rows to answer a question about 12 million. No engine fixes that."

**What made this work.**

- **Meera asked about the data the job touches,** not the size of the archive.
- **The storage layout was the real problem;** the engine was a symptom.
- **They kept Spark where it earned its place** instead of treating the choice as ideological.

---

## Project: three questions, three engines

**Goal:** answer the plant team's questions on the sensor data, choose an engine with evidence, and be able to defend the choice.

### Tools you'll need

- **Python** in the book's virtual environment with `pyspark` and `polars` (section 48.0), `duckdb` (Chapter 45), and `pandas` and `pyarrow` (Chapter 18); **Java 17, 21, or 25** (check with `java -version`).
- **The Chapter 48 companion folder** (`companion/ch48/`), copied to `work/ch48`: `make_sensor_data.py` (generates the dataset) and `time_three_tools.py` (the timing script from section 48.6). Run Python from that folder.
- **Versions used for the outputs shown:** Python 3.11.15, PySpark 4.2.0, Polars 1.44.2, DuckDB 1.5.6, PyArrow 25.0.1, pandas 3.0.6, OpenJDK 21.0.10. The timings came from a shared test machine with 4 cores, with each engine given two.
- **Spark's web interface** at `http://localhost:4040` while a session runs (section 48.5).

**Option A: Riverstone.** Use the generated sensor dataset.

**Option B: your own data.** Any dataset of at least a few million rows that you're allowed to use, stored as Parquet.

**Steps**

1. **Generate the data** and record its size on disk and its row count.
2. **Answer three questions** in DuckDB, Polars, and PySpark, and confirm all three give identical numbers:
   a. For each machine, the share of readings above 200 °C in December.
   b. For each plant and week, total units made and scrap percentage.
   c. The ten machine-days with the highest scrap rate, with at least 5,000 readings that day.
3. **Time each engine** on each question, three runs, and report the fastest. State your hardware.
4. **Read the Spark plan** for question (c). Write down every exchange and what causes it.
5. **Make one query faster** by pruning, broadcasting, or reducing shuffles. Show the plan before and after, and the timing change.
6. **Break something on purpose:** filter on `F.year(F.col("reading_ts"))` instead of the partition column, and show what happens to `PartitionFilters` and to the timing.
7. **Write a recommendation** of half a page: which engine for this report, at what data size you'd change your mind, and what it would cost to be wrong.

**Stretch goals**

- Repeat with the dataset ten times larger (change `DAYS` in the generator) and see whether the ranking changes.
- Write the output of question (b) as Parquet partitioned by week, and measure how much faster the next read gets.
- Introduce skew by duplicating one machine's rows, and show what adaptive query execution does about it.

---

## Recap

- **Most "big data" isn't.** Ask what the job actually reads, after filtering.
- Spark needs **Java** and the `pyspark` library; a **SparkSession** in **local mode** runs it on your own cores.
- A distributed engine splits data into **partitions**, runs work on **workers**, and coordinates from the **driver**.
- **Narrow** operations stay inside a partition and are cheap; **wide** operations need a **shuffle** and are expensive. Every `Exchange` in a plan is a shuffle, and **`explain()`** shows the plan.
- **Lazy execution** means nothing runs until an **action**; the plan is built from the whole query.
- **Partition pruning** (folders skipped) and **column pruning** (columns not read) remove work before it starts; check `PartitionFilters` and `ReadSchema`.
- A **broadcast join** sends a small table to every worker so the big one never moves.
- **Skew** makes one worker do most of the work; adaptive execution, salting, or special-casing the hot key fix it.
- **Polars** builds queries from expressions, and its lazy mode plans them as Spark does.
- On one machine, at 19.87 million rows, **DuckDB (213 ms) and Polars (262 ms) beat Spark (1,212 ms)**; the numbers are machine-specific, and Spark's advantage appears when data outgrows one machine.
- Avoid `collect()` on large results, thousands of tiny files, careless caching, and Python UDFs in hot paths.
- Learn SQL and one single-machine engine deeply, understand Spark's model, and add a platform when a job needs it.

---

## Key terms

distributed computing · Java runtime · `JAVA_HOME` · SparkSession · partition · worker (executor) · driver · narrow operation · wide operation · shuffle · exchange · lazy execution · action · query plan · physical plan · `explain()` · adaptive query execution (AQE) · Hive-style partitioning · partition pruning · column pruning · pushed filter · broadcast join · sort-merge join · skew · salting · cache · collect · small files problem · user-defined function (UDF) · local mode · Spark web interface · DuckDB · Polars · expression · lazy frame · PySpark · Databricks · EMR · Dataproc · Trino · Ray

*(All terms are defined in the Glossary, Appendix A.)*

---

## Check yourself

- [ ] You can install Java and PySpark, start a SparkSession, and fix the usual start-up errors.
- [ ] You can argue, with numbers, whether a dataset needs a distributed engine.
- [ ] You can explain partitions, workers, the driver, and narrow versus wide operations.
- [ ] You can say what a shuffle is and why it dominates cost.
- [ ] You can write PySpark to read, filter, group, join, and write Parquet.
- [ ] You can explain lazy execution and name three actions.
- [ ] You can read a physical plan with `explain()`, find every exchange, and say what caused each one.
- [ ] You can prove partition pruning and column pruning are happening.
- [ ] You can turn a sort-merge join into a broadcast join, and explain when not to.
- [ ] You can describe skew, its symptoms, and three fixes.
- [ ] You can write a simple Polars query with expressions, eagerly and lazily.
- [ ] You can compare engines on correctness first, then speed, and state your hardware.
- [ ] You can list five ways a Spark job surprises people, including `collect()` and small files.
- [ ] You can place Databricks, EMR, Trino, and Ray on the map.

---

## Exercises

### Warm-up

1. Classify each operation as narrow or wide: (a) `filter`; (b) `groupBy(...).agg(...)`; (c) `select`; (d) `join` on a key; (e) `withColumn` with arithmetic; (f) `orderBy`.
2. In the plan in section 48.4, name every line that causes data to move between workers, and say which column it partitions by.
3. A colleague's job on 4 million rows takes 90 seconds in Spark on their laptop and 3 seconds in DuckDB. Give two reasons, and say what you'd recommend.

### Core

4. Write the PySpark for: for each machine in December, the number of readings, the share above 200 °C as a percentage, and the total units made, ordered by that share, highest first.
5. Using the plan from section 48.5's pruning example, explain the difference between `PartitionFilters` and `PushedFilters`. Which one would disappear if the data weren't partitioned by date, and what would the query then cost?
6. The join in section 48.5 shuffled 19.87 million rows to match them with 3. Explain why Spark did that when broadcasting was switched off, and why `F.broadcast()` is still worth writing even when the automatic threshold would catch it.
7. `spark.sql.shuffle.partitions` is 200 by default. Explain what goes wrong with 200 partitions on a one-core machine, and what goes wrong with 8 partitions on a 200-core cluster.
8. From the timings in section 48.6, calculate how many times slower Spark was than DuckDB at each data size. What does the change between the two sizes suggest about where the crossover might be?
9. A job writes its output with `df.write.parquet(...)` after a group-by, and produces 200 files of 400 KB each. Explain the problem this creates for tomorrow's reader, and write the one line that fixes it.

### Stretch

10. A join between readings and a 2 GB table of machine settings is slow, and the plan shows `SortMergeJoin` with two exchanges. Broadcasting isn't possible. List three other things you'd try, in order, and what you'd check in the plan after each.
11. Half of Riverstone's readings come from one high-volume machine, and a join on `machine_id` has one task that never finishes. Describe the salting fix in detail, including what happens to the small side, and what you'd check afterwards.
12. Design the storage layout for the sensor archive so that the weekly report in "In the real world" prunes correctly: partition columns, file sizes, sort order inside files, and what you'd do about the existing files.

### Think about it (no code needed)

13. Spark appears in far more job descriptions than most teams' daily work needs. How would you prepare for those interviews without building a platform you don't need? Be specific about what you'd learn and what you'd build.
14. "The cluster wasn't wrong; the storage layout was." Explain to a non-technical manager why moving the same data into a bigger machine can fail to fix a slow report, using the votes-and-boxes analogy from the start of the chapter.

---

## Answers

**1.** Narrow: (a) `filter`, (c) `select`, (e) `withColumn` with arithmetic. Wide: (b) `groupBy().agg()`, (d) `join` on a key (unless it's broadcast, in which case the big side isn't shuffled), (f) `orderBy` (a global sort needs data moved so ranges are in order).

**2.** One line: `Exchange hashpartitioning(machine_id, 8)`. It partitions by `machine_id`: it redistributes rows by a hash of `machine_id` into 8 partitions, so every row for a machine lands in the same partition. `ENSURE_REQUIREMENTS` means the aggregate above it needs that arrangement. The `HashAggregate … partial_sum` below it is what keeps the shuffle small: only partial totals cross, not raw rows.

**3.** Two reasons: (1) **Overhead:** Spark starts a Java process, plans, and coordinates tasks; at 4 million rows that setup is most of the runtime, while DuckDB runs in the same process. (2) **Too many shuffle partitions:** with the default 200, the job creates hundreds of tiny tasks on a machine with a handful of cores. Recommendation: keep it in DuckDB (or Polars), and revisit only if the data grows past what the machine can hold. If Spark must be used, set `spark.sql.shuffle.partitions` to a few times the core count.

**4.** A sample answer:

In [ ]:
december = readings.filter(F.col("reading_date").between("2025-12-01", "2025-12-31"))

answer = (december.groupBy("machine_id")
          .agg(F.count("*").alias("readings"),
               F.round(F.avg((F.col("temperature_c") > 200).cast("int")) * 100, 2).alias("hot_pct"),
               F.sum("units_made").alias("units"))
          .orderBy(F.col("hot_pct").desc(), "machine_id"))
answer.show(5)

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

**What the chapter shows:**

```
+----------+--------+-------+------+
|machine_id|readings|hot_pct| units|
+----------+--------+-------+------+
|      M-14|  267840|  43.88|803351|
|      M-24|  267840|  43.88|803690|
|      M-19|  267840|  43.82|803423|
|      M-23|  267840|  43.73|804166|
|      M-18|  267840|  43.71|803467|
+----------+--------+-------+------+
only showing top 5 rows
```

The filter is on the partition column, so only December's folders are read; `(condition).cast("int")` then averaged is the usual way to turn a share into a percentage. `between(a, b)` includes both ends, `.desc()` sorts from highest to lowest, and `"machine_id"` breaks ties, so machines with the same share always come out in the same order.

**5.** **`PartitionFilters`** are applied to the *folder structure*: Spark decides which `reading_date=` folders to open at all, so unmatched files are never read. **`PushedFilters`** are applied *inside* the Parquet files as they're read, using each file's own row-group statistics to skip chunks where the values are clustered, and then row by row. Without date partitioning, `PartitionFilters` would be empty; the date condition would become an ordinary pushed filter, Spark would open all 92 days of files, and the query would read roughly 92 times as much data (though Parquet statistics would still let it skip some row groups if the files were sorted by time).

**6.** With broadcasting switched off, Spark's only general way to match rows is to put equal keys in the same partition, which means hash-partitioning **both** sides by `machine_id`: the 19.87 million-row side gets written to shuffle files (on a cluster, also sent over the network) and read back. `F.broadcast()` is still worth writing because Spark's automatic decision rests on an **estimate** of the small side's size, and estimates go wrong after filters, joins, or aggregations, especially with no table statistics. Being explicit also documents the intent for the next reader. The caution: broadcasting a table that isn't actually small can run the driver and every worker out of memory.

**7.** With **200 partitions on one core**, every shuffle produces 200 small tasks that run more or less one after another, each with scheduling overhead and a tiny file; the job spends most of its time coordinating. With **8 partitions on 200 cores**, only 8 tasks exist, so at most 8 cores do anything and the other 192 sit idle, and each task handles a large slice that may not fit comfortably in memory. The rule of thumb is a few times the total core count, adjusted so each partition holds roughly 100–200 MB.

**8.** At 216,000 rows: 252.3 ÷ 6.8 ≈ **37 times slower**. At 19.87 million rows: 1,211.8 ÷ 213.0 ≈ **5.7 times slower**. The ratio fell sharply as the data grew, which is what you'd expect if a fixed overhead is being spread over more work. (Work from the milliseconds, not rounded seconds: at this speed, rounding to 0.01 s would turn 6.8 ms into 0.01 s and the ratio into nonsense.) Extrapolating a ratio isn't a prediction, but the direction suggests the crossover lies well beyond this size, and on one machine it may never arrive: DuckDB and Polars speed up with more cores just as Spark does, and Spark's real advantage is adding machines, which this setup can't do.

**9.** 200 files of 400 KB means the next reader opens 200 files to read 80 MB, and each open costs a round trip, especially on cloud storage: the "small files problem". Metadata handling and task scheduling then dominate the read. The fix is to reduce the number of output partitions before writing, for example:

In [ ]:
df.coalesce(1).write.mode("overwrite").parquet("mart/weekly_scrap")

> **Not run here.** This cell depends on a cell above that could not run here, so its names were never created. Run it on your own machine, with the setup the chapter describes, and you should see what the chapter shows.

> **Not run here.** The chapter marks this block as illustration rather than something to execute, usually because it needs a service, a key or a file this notebook does not set up. Read it, do not run it.

`coalesce(n)` merges partitions without a shuffle (cheaper); `repartition(n)` reshuffles to exactly `n` and balances them. For larger outputs, aim for files of 128–512 MB rather than a single file.

**10.** In order: (1) **Filter and select earlier**, so less of the 2 GB reaches the join, then check the plan shows the filters below the exchanges and a smaller `ReadSchema`. (2) **Pre-partition or bucket both tables on the join key** in storage, so the join can happen without a shuffle; check for the disappearance of one or both exchanges. (3) **Reduce the join itself:** aggregate the settings table to one row per key first, or join on a narrower key, then check the estimated sizes in the plan. If it's still slow, look for **skew** (one long task in the Spark UI) and apply the fixes in 48.5, and only then consider more workers.

**11.** Add a **salt** to spread the hot key. On the **big** side, add a column `salt = floor(rand() * N)` for some N, say 8. On the **small** side, replicate every row N times, once per salt value (`explode` over an array of 0…N−1), so that whatever salt a big row got, there's a matching small row. Join on `[machine_id, salt]`. The hot machine's rows now spread across 8 partitions instead of one, so 8 workers share the work, at the cost of the small side being 8 times larger. Afterwards, check in the Spark UI that the longest task in that stage is close to the median task, and confirm row counts are unchanged, as section 48.5 did, since a mistake in replicating the small side silently duplicates or drops rows.

**12.** Partition by **date** (`reading_date=YYYY-MM-DD`), because every report filters by time; that alone lets the weekly report read 7 folders out of thousands. Inside each day, **sort by `machine_id`** (and write with statistics), so Parquet row-group pruning can skip most of a file when a report asks about a few machines. At Riverstone's volume a day is only about 10 MB (about 1.7 million readings), so aim for **one file per day**, never one per machine; if the file count grows into the thousands, partition by month instead, or rely on regular compaction (Chapter 49). The 128–512 MB target applies once a partition is that large. For the existing archive, run a one-off **rewrite job**: read the old layout, write the new one partitioned by date, verify counts and totals per day match the old files (Chapter 47's reconciliation), then switch readers over and keep the old files until the next cycle has proved the new layout works. Machine-based partitioning should be dropped: it produced thousands of small files and prunes nothing for time-based reports.

**13.** A specific answer: learn Spark's **model** rather than its operations catalogue: partitions, narrow versus wide, shuffles, joins, skew, and how to read a plan. Build **one** honest project: a few tens of millions of rows in local mode, with a plan you can explain, a broadcast join before and after, and measured timings against DuckDB. Read the Spark UI for a real job. Then be ready to say, in an interview, when you would *not* use Spark, and why, with your own numbers. That combination (model, one real project, and judgment about when it's the wrong tool) answers most interview questions, and it's honest about what you've done. Anything beyond that is best learned on a job that actually has the data.

**14.** Using the analogy: the report is asking a question about one district's votes, but the ballots were sorted into boxes by candidate rather than by district. So whichever way you organize the counting officers, every box still has to be opened to find that district's ballots. Hiring more officers (a bigger machine or a cluster) means the opening happens in parallel, but the total work stays enormous, and coordination adds its own cost. If instead the ballots are stored in one box per district, a single officer answers the question in minutes by opening one box. The lesson for the manager: the way the data is organized decides how much work a question needs; the size of the machine only decides how fast that work gets done.

---

## Where this leads

- **Chapter 49, Storage, Warehouses & Lakehouses,** explains the layouts this chapter depends on: partitioning, file sizes, table formats with transactions and time travel, and cloud warehouse costs.
- **Chapter 50, Streaming & Real-Time,** uses the same sensor data as a never-ending stream, with Spark Structured Streaming.
- **Chapter 47** applies: tests and reconciliation matter more at this scale, not less, and they must be cheap to run.
- **Chapter 52, The Cloud, Containers & Infrastructure as Code,** covers packaging and running jobs like these in containers and in the cloud, instead of on your laptop.
- **Part 8:** "how would you process X" design cases, and questions on partitioning and scale, come up in the data engineering bank, Chapter 77 (section 77.5).